# Deploy / update SEI_Dashboard in this Fabric workspace

Creates **or updates** the **SEI_Dashboard** semantic model and report from the files in the public GitHub repo.
No Git integration, Azure DevOps or Power BI Desktop needed. Just **Run all** (about 2-3 minutes).
Re-run it any time the repo changes.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import time
import requests
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.report import create_report_from_reportjson, update_report_from_reportjson

REPO = "https://raw.githubusercontent.com/NirmalKumar31/social-engagement-intelligence-pro/main/powerbi"
NAME = "SEI_Dashboard"
nocache = {"Cache-Control": "no-cache"}

bim = requests.get(f"{REPO}/SEI_Dashboard.SemanticModel/model.bim", headers=nocache, timeout=60).json()
report_json = requests.get(f"{REPO}/SEI_Dashboard.Report/report.json", headers=nocache, timeout=60).json()
print(f"Loaded model: {len(bim['model']['tables'])} tables, {len(bim['model']['relationships'])} relationships")
print(f"Loaded report: {len(report_json['sections'])} pages")

## 1. Semantic model (create or update)

In [ ]:
if NAME in fabric.list_datasets(mode="rest")["Dataset Name"].values:
    labs.update_semantic_model_from_bim(dataset=NAME, bim_file=bim)
else:
    labs.create_semantic_model_from_bim(dataset=NAME, bim_file=bim)

for _ in range(30):   # wait until the service lists it
    if NAME in fabric.list_datasets(mode="rest")["Dataset Name"].values:
        print("semantic model is ready")
        break
    time.sleep(5)

## 2. Load the data

First run only: if this fails with a **credentials** error, open the semantic model **... → Settings → Data source credentials → Edit (pencil) → Anonymous / Public → Sign in**, then re-run this cell.

In [ ]:
labs.refresh_semantic_model(dataset=NAME)

## 3. Report (create or update)

In [ ]:
if NAME in fabric.list_reports()["Name"].values:
    update_report_from_reportjson(report=NAME, report_json=report_json)
else:
    create_report_from_reportjson(report=NAME, dataset=NAME, report_json=report_json)
print("Done. Open the SEI_Dashboard report (refresh the browser tab if it was already open).")